# 一个用于构建 API 的现代、快速（高性能）的 web 框架
# 快速构建 api，异步框架（django和flask旧版本不是异步框架） I
# 易于使用和学习
# 自动生成的交互式文档
# ```
# 前后端分离
## 1. 前后端不分离（传统方式）

# **图中左侧部分表达的意思**：
# - 访问 `127.0.0.1:8000/index.html` 直接拿到完整页面
# - URL 路由直接返回 HTML 页面（如 `def login: return login.html`）
# - 后端负责渲染整个页面，前端只是点缀
# **通俗理解**：
# > 前端和后端**写在一个项目里**，后端直接把完整的 HTML 页面返回给浏览器。

# **流程**：
# ```
# 浏览器请求 → 后端服务器 → 查数据库 → 渲染HTML → 返回完整页面
# ```

# **特点**：
# - 页面刷新靠后端控制
# - 典型框架：Django、Flask（旧版本）、ThinkPHP
# - 缺点：前后端代码混在一起，改前端也要动后端

# ## 2. 前后端分离（现代方式）

# **图中右侧部分表达的意思**：
# - Vue 等前端框架单独负责页面（`main.js`、`index.vue`）
# - FastAPI 只提供 API 接口（`app.update`、`app.delete`）
# - 前后端通过 API 通信，各干各的

# **通俗理解**：
# > 前端是一个独立项目（Vue/React），后端是另一个独立项目（FastAPI），两者通过 API 接口交换数据。

# **流程**：
# ```
# 浏览器请求 → 前端服务器(Vue) → 返回HTML+JS
#                 ↓
#          调用API ← FastAPI后端 → 数据库
# ```

# **特点**：
# - 页面渲染在前端完成
# - 后端只返回 JSON 数据，不关心页面长什么样
# - 一套后端 API 可以同时给 Web、App、小程序用

# ## 对比总结

# | 维度 | 不分离 | 分离 |
# |------|--------|------|
# | 返回内容 | HTML 页面 | JSON 数据 |
# | 谁渲染页面 | 后端 | 前端 |
# | 开发方式 | 前后端代码混在一起 | 两个独立项目 |
# | 适合场景 | 纯网页、后台管理系统 | App、小程序、复杂前端 |
# | 典型框架 | Flask旧版、Django | FastAPI + Vue/React |

# ## 图中右下角那个 `zhangsan 123`

# 这可能是**示例数据**或**测试账号**，表示数据库中存有用户 `zhangsan`，密码 `123`，与前面的 API 示例呼应。

# **总结**：你学的 FastAPI 天然适合**前后端分离**模式——它只负责提供 API 接口，返回 JSON 数据，前端（Vue/React/小程序）自己去调用这些接口并展示页面。

# 根据动作写路由
# /addstudent/     # 添加学生
# /updatestudent/  # 更新学生
# /deletestudent/  # 删除学生
# /selectstudent/  # 查询学生
# 把动作写入到url中
# 路由就是根据 URL 路径，找到对应的处理函数。
# 改进

In [ ]:
# ============================================================
# 【报错原因】原来的代码全部是注释(#开头)，Python解释器读到的是空文件，没有任何可执行语句
# 以下为完整的 FastAPI CRUD 代码，运行前先执行: pip install fastapi uvicorn
# ============================================================

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import Optional
import uvicorn

# 【报错原因】如果没有创建 app 实例，后续 @app.get() / @app.post() 装饰器找不到 app 变量，会报 NameError
app = FastAPI(title="学生管理系统 API")

# === 数据模型 ===
# 【报错原因】FastAPI 必须用 Pydantic BaseModel 定义数据结构
# 如果直接传普通 dict，FastAPI 无法自动校验字段类型，前端传错数据不会报错
class Student(BaseModel):
    id: int = Field(description="学生ID")
    name: str = Field(min_length=1, max_length=50)
    age: int = Field(ge=1, le=150)
    grade: str = Field(default="一年级")

class StudentUpdate(BaseModel):
    # 【报错原因】更新操作如果用同一个 Student 模型，所有字段都会变成必填，无法实现部分更新
    # Optional 让这些字段可选，只更新用户传了的字段
    name: Optional[str] = Field(None, min_length=1, max_length=50)
    age: Optional[int] = Field(None, ge=1, le=150)
    grade: Optional[str] = None

# === 模拟数据库 ===
# 【报错原因】没有数据存储的话，CRUD 操作无法验证效果，用内存字典模拟即可学习测试
fake_db: dict[int, Student] = {}

# === 查询学生 ===
@app.get("/selectstudent/")
async def select_student(id: int):
    """根据ID查询学生: http://127.0.0.1:8000/selectstudent/?id=1"""
    student = fake_db.get(id)
    if student is None:
        # 【报错原因】找不到资源必须返回 404，如果 return None 前端无法判断是"没数据"还是"接口挂了"
        raise HTTPException(status_code=404, detail=f"学生 ID={id} 不存在")
    return student

@app.get("/selectstudent/all")
async def select_all_students():
    """查询全部学生"""
    return list(fake_db.values())

# === 添加学生 ===
@app.post("/addstudent/")
async def add_student(student: Student):
    """添加学生: POST 请求体 {"id":1,"name":"张三","age":20,"grade":"二年级"}"""
    if student.id in fake_db:
        # 【报错原因】不检查重复直接覆盖会导致数据丢失，应该拒绝并提示用户
        raise HTTPException(status_code=400, detail=f"学生 ID={student.id} 已存在")
    fake_db[student.id] = student
    return {"message": "添加成功", "student": student}

# === 更新学生 ===
@app.put("/updatestudent/")
async def update_student(id: int, student_update: StudentUpdate):
    """更新学生: PUT 请求体 {"name":"张三丰","age":21}"""
    existing = fake_db.get(id)
    if existing is None:
        raise HTTPException(status_code=404, detail=f"学生 ID={id} 不存在")
    # 【报错原因】model_dump(exclude_unset=True) 只提取用户实际传了的字段
    # 如果不用这个，None 值会覆盖原有数据，导致字段被意外清空
    update_data = student_update.model_dump(exclude_unset=True)
    updated_student = existing.model_copy(update=update_data)
    fake_db[id] = updated_student
    return {"message": "更新成功", "student": updated_student}

# === 删除学生 ===
@app.delete("/deletestudent/")
async def delete_student(id: int):
    """删除学生: http://127.0.0.1:8000/deletestudent/?id=1"""
    student = fake_db.pop(id, None)
    if student is None:
        raise HTTPException(status_code=404, detail=f"学生 ID={id} 不存在")
    return {"message": "删除成功", "student": student}

# === 启动说明 ===
# 【报错原因】Notebook 中不能直接调用 uvicorn.run(app)，它会阻塞单元格导致无法执行后续代码
# 正确做法: 运行完本单元格后，在终端执行:
#   python "# fastapi.py"
# 然后浏览器访问 http://127.0.0.1:8000/docs 查看自动生成的接口文档